In [20]:
# ===========================
# 01. IMPORTS
# ===========================
from pyspark.sql.functions import col, current_timestamp


# ===========================
# 02. PARAMETERS
# ===========================

# Source (Silver Delta).
source_catalog = "lab_catalog_silver"
source_schema  = "digital_insurance"
source_table   = "tb_sinistro"

# Target (Gold external catalog).
target_catalog = "lab_ailakehouse_gold"
target_schema  = "lab_digitalinsurance_aidp"
target_table   = "tb_catastrofe_alerta"

# Build derived table names.
source_full_table_name = f"{source_catalog}.{source_schema}.{source_table}"
target_full_table_name = f"{target_catalog}.{target_schema}.{target_table}"

# Streaming checkpoint path.
gold_checkpoint_path = (
    f"/Volumes/{source_catalog}/{source_schema}/checkpoints/{target_table}_checkpoint_gold/"
)

# ===========================
# 03. DETECTION PARAMETERS
# ===========================

# Sliding window configuration.
WINDOW_DURATION = "1 hour"
SLIDE_DURATION  = "15 minutes"

# Watermark delay.
WATERMARK_DELAY = "2 hours"

# Alert thresholds.
MIN_SINISTROS_ALERTA = 5
VALOR_ALERTA         = 100000.0

In [21]:
# ===========================
# 03. BUSINESS QUERY
# ===========================

# Filter and enrich incoming claims with policy information.
def sinistros_validos(df_sinistro):
    # Register the streaming DataFrame as a temporary view.
    df_sinistro.createOrReplaceTempView("stream_sinistro")

    # Apply business rules and generate the geographic cell.
    return spark.sql(f"""
        SELECT
            CAST(s.sinistro_id AS BIGINT)                                        AS sinistro_id,
            CAST(s.apolice_id  AS BIGINT)                                        AS apolice_id,
            to_timestamp(s.data_hora_abertura, 'yyyy-MM-dd HH:mm:ss[.SSSSSSSSS]') AS data_hora_abertura,
            s.tipo_sinistro,
            CAST(s.valor_estimado AS DOUBLE)                                     AS valor_estimado,
            CAST(s.latitude  AS DOUBLE)                                          AS latitude,
            CAST(s.longitude AS DOUBLE)                                          AS longitude,
            CONCAT(
                CAST(ROUND(CAST(s.latitude  AS DOUBLE), 2) AS STRING), ':',
                CAST(ROUND(CAST(s.longitude AS DOUBLE), 2) AS STRING)
            )                                                                    AS geo_cell,
            COALESCE(CAST(a.valor_cobertura AS DOUBLE), 0.00)                    AS valor_cobertura
        FROM stream_sinistro s
        LEFT JOIN {source_catalog}.{source_schema}.tb_apolice a
               ON CAST(s.apolice_id AS BIGINT) = CAST(a.apolice_id AS BIGINT)
        WHERE CAST(s.sinistro_id AS BIGINT) > 0
          AND CAST(s.apolice_id  AS BIGINT) > 0
          AND s.data_hora_abertura IS NOT NULL
          AND to_timestamp(s.data_hora_abertura, 'yyyy-MM-dd HH:mm:ss[.SSSSSSSSS]')
              > TIMESTAMP '2000-01-01 00:00:00'
          AND CAST(s.latitude  AS DOUBLE) BETWEEN -90  AND 90
          AND CAST(s.longitude AS DOUBLE) BETWEEN -180 AND 180
          AND NOT (CAST(s.latitude AS DOUBLE) = 0 AND CAST(s.longitude AS DOUBLE) = 0)
    """)

In [22]:
# ===========================
# 04. STATEFUL AGGREGATION
# ===========================

# Aggregate claims by geographic cell using a sliding time window.
def agrega_catastrofe(df_validos):
    # Apply the watermark before the windowed aggregation.
    df_validos.withWatermark("data_hora_abertura", WATERMARK_DELAY) \
              .createOrReplaceTempView("sinistros_com_watermark")

    # Calculate aggregated metrics and alert indicators.
    return spark.sql(f"""
        SELECT
            window.start AS janela_inicio,
            window.end   AS janela_fim,
            geo_cell,
            qtd_sinistros,
            valor_total_estimado,
            valor_total_cobertura,
            lat_centroide,
            lon_centroide,
            primeiro_sinistro,
            ultimo_sinistro,
            (qtd_sinistros / 20.0 * 0.60
             + valor_total_estimado / 1000000.0 * 0.40) AS severidade,
            CAST(
                (qtd_sinistros >= {MIN_SINISTROS_ALERTA}
                 AND valor_total_estimado >= {VALOR_ALERTA}) AS INT
            ) AS is_catastrofe
        FROM (
            SELECT
                window(data_hora_abertura, '{WINDOW_DURATION}', '{SLIDE_DURATION}') AS window,
                geo_cell,
                COUNT(1)                AS qtd_sinistros,
                SUM(valor_estimado)     AS valor_total_estimado,
                SUM(valor_cobertura)    AS valor_total_cobertura,
                AVG(latitude)           AS lat_centroide,
                AVG(longitude)          AS lon_centroide,
                MIN(data_hora_abertura) AS primeiro_sinistro,
                MAX(data_hora_abertura) AS ultimo_sinistro
            FROM sinistros_com_watermark
            GROUP BY
                window(data_hora_abertura, '{WINDOW_DURATION}', '{SLIDE_DURATION}'),
                geo_cell
        )
    """)

In [23]:
# ===========================
# 05. TARGET
# ===========================

# Target table definition.
TARGET_DDL = f"""
    CREATE TABLE {target_full_table_name} (
        janela_inicio          TIMESTAMP,
        janela_fim             TIMESTAMP,
        geo_cell               STRING,
        qtd_sinistros          BIGINT,
        valor_total_estimado   DOUBLE,
        valor_total_cobertura  DOUBLE,
        lat_centroide          DOUBLE,
        lon_centroide          DOUBLE,
        primeiro_sinistro      TIMESTAMP,
        ultimo_sinistro        TIMESTAMP,
        severidade             DOUBLE,
        is_catastrofe          INT,
        dt_atualizacao         TIMESTAMP
    )
"""

# Current alerts view.
TARGET_VIEW = f"{target_catalog}.{target_schema}.vw_{target_table}_atual"
VIEW_DDL = f"""
    CREATE OR REPLACE VIEW {TARGET_VIEW} AS
    SELECT janela_inicio, janela_fim, geo_cell, qtd_sinistros,
           valor_total_estimado, valor_total_cobertura,
           lat_centroide, lon_centroide,
           primeiro_sinistro, ultimo_sinistro,
           severidade, is_catastrofe, dt_atualizacao
    FROM (
        SELECT h.*,
               ROW_NUMBER() OVER (
                   PARTITION BY janela_inicio, janela_fim, geo_cell
                   ORDER BY dt_atualizacao DESC
               ) AS rn
        FROM {target_full_table_name} h
    )
    WHERE rn = 1
      AND is_catastrofe = 1
"""


def build_target():
    """Create the target table and view."""
    spark.sql(TARGET_DDL)
    spark.sql(VIEW_DDL)


# Output columns in the target table order.
TARGET_COLS = [
    "janela_inicio", "janela_fim", "geo_cell", "qtd_sinistros",
    "valor_total_estimado", "valor_total_cobertura",
    "lat_centroide", "lon_centroide",
    "primeiro_sinistro", "ultimo_sinistro",
    "severidade", "is_catastrofe", "dt_atualizacao",
]


def insert_alertas(batch_df, batch_id):
    """Insert active alerts into the target table."""
    alertas = (
        batch_df
        .where(col("is_catastrofe") == 1)
        .withColumn("dt_atualizacao", current_timestamp())
        .select(*TARGET_COLS)
    )

    # Skip empty micro-batches.
    if alertas.isEmpty():
        return

    alertas.write.mode("append").insertInto(target_full_table_name)

In [24]:
# ===========================
# 06. TARGET BOOTSTRAP
# ===========================

# Create the target table and view if they do not exist.
if not spark.catalog.tableExists(target_full_table_name):
    print("Target table does not exist. Creating target table and view...")
    build_target()
    print("Target objects created successfully.")
else:
    print("Target table already exists. Skipping bootstrap.")

Target table already exists. Skipping bootstrap.


In [28]:
# =================================================================================
# 07. STREAM PROCESSING readStream Delta -> filtro -> agregacao stateful -> Oracle
# =================================================================================

# Get the physical location of the source Delta table.
table_path = (
    spark.sql(f"DESCRIBE DETAIL {source_full_table_name}")
         .select("location")
         .first()["location"]
)

# Create a streaming DataFrame from the source Delta table.
sinistros_stream = (
    spark.readStream
         .format("delta")
         .load(table_path)
)

# Apply business rules and stateful aggregation.
df_validos    = sinistros_validos(sinistros_stream)
df_catastrofe = agrega_catastrofe(df_validos)

# Process each micro-batch and write active alerts to the target table.
stream = (
    df_catastrofe.writeStream
        .outputMode("update")
        .option("checkpointLocation", gold_checkpoint_path)
        .foreachBatch(insert_alertas)
        .trigger(availableNow=True)  # Change to processingTime="1 minute" for continuous execution every minute.
    	.queryName(f"gold_{target_full_table_name}")
        .start()
)

# Wait for the streaming job to complete.
stream.awaitTermination()